## **Read from bronze and write to a table in silver catalog with an additional column.** 

In [1]:
bronze_catalog = "bronze_supplier"
silver_catalog = "silver_supplier"
bronze_schema  = "supplier_schema"
silver_schema  = "supplier_schema"
bronze_table_csv = "bronze_supplier_basic_csv"
silver_table_csv = "silver_supplier_basic_csv"
silver_supplier_basic_Cont_csv = "silver_supplier_basic_Cont_csv"


## Create silver catalog and schema if they do not exist. ##

In [1]:
spark.sql("create catalog if not exists silver_supplier") 
spark.sql("create schema if not exists silver_supplier.supplier_schema") 




DataFrame[]

In [1]:

## Read the table from the bronze into a data frame.
silver_df = spark.read.table(f"{bronze_catalog}.{bronze_schema}.{bronze_table_csv}")

In [1]:
## display the data frame 
silver_df.show()

+--------------------+-------------+--------------------+-------------------+
|       supplier_name|supplier_type|    supplier_country|      supplier_city|
+--------------------+-------------+--------------------+-------------------+
|24/7 Community Ho...|         NULL|       United States|           Glendale|
|     A1 Juice Supply|         NULL|       United States|             Aurora|
|            ABC Bank|         NULL|       United States|       Redwood City|
|      ABC Consulting|         NULL|           Australia|             Sydney|
|              Abbott|         NULL|       United States|        Lake Forest|
|       Adams Leasing|         NULL|       United States|          Greenwood|
|       Advanced Corp|         NULL|       United States|          BALTIMORE|
|        Aesculup Inc|         NULL|       United States|South San Francisco|
|               Ajeer|         NULL|        Saudi Arabia|             Riyadh|
|Al Safa General C...|         NULL|United Arab Emirates|       

## read from Bronze and create a new data frame for Silver ## 

In [1]:
from pyspark.sql.functions import col, current_timestamp
from pyspark.sql.types import IntegerType, StringType

# Apply renaming, casting, and select only renamed columns
silver_augmented_df = silver_df.select(
    col("supplier_name").cast(StringType()).alias("supplier_name_silver"),
    col("supplier_type").cast(StringType()).alias("supplier_type_silver"),
    col("supplier_country").cast(StringType()).alias("supplier_country_silver"),
    col("supplier_city").cast(StringType()).alias("supplier_city_silver"),
        ).withColumn("ingestion_date", current_timestamp())

In [1]:

# write the data frame to a file in silver catalog with an additonal column and new column names

silver_augmented_df.write.mode("overwrite").format("csv").saveAsTable(f"{silver_catalog}.{silver_schema}.{silver_table_csv}")

In [1]:
%sql
select * from silver_supplier.supplier_schema.silver_supplier_basic_csv

In [1]:
spark.sql("select * from silver_supplier.supplier_schema.silver_supplier_basic_csv  ").show(10, False)

+-----------------------------------+--------------------+-----------------------+--------------------+-----------------------+
|supplier_name_silver               |supplier_type_silver|supplier_country_silver|supplier_city_silver|ingestion_date         |
+-----------------------------------+--------------------+-----------------------+--------------------+-----------------------+
|24/7 Community Hospital Group      |NULL                |United States          |Glendale            |2025-12-10 17:05:26.073|
|A1 Juice Supply                    |NULL                |United States          |Aurora              |2025-12-10 17:05:26.073|
|ABC Bank                           |NULL                |United States          |Redwood City        |2025-12-10 17:05:26.073|
|ABC Consulting                     |NULL                |Australia              |Sydney              |2025-12-10 17:05:26.073|
|Abbott                             |NULL                |United States          |Lake Forest         |2

## Drop the table since it is not needed anymore and create a new table with different columns ##

In [1]:
%sql
drop table if exists silver_supplier.supplier_schema.silver_supplier_basic_Cont_csv

OK

In [1]:

spark.sql("create table if not exists silver_supplier.supplier_schema.silver_supplier_basic_Cont_csv (supplier_name_silver  string,  supplier_type_silver  STRING, supplier_country_silver  STRING, supplier_city_silver STRING, continent STRING )")

extended_df = spark.read.table(f"{silver_catalog}.{silver_schema}.{silver_supplier_basic_Cont_csv}")



In [1]:
spark.sql("select * from silver_supplier.supplier_schema.silver_supplier_basic_cont_csv  ").show(10, False)

+--------------------+--------------------+-----------------------+--------------------+---------+
|supplier_name_silver|supplier_type_silver|supplier_country_silver|supplier_city_silver|continent|
+--------------------+--------------------+-----------------------+--------------------+---------+
+--------------------+--------------------+-----------------------+--------------------+---------+



In [1]:
%sql
/*  check to make sure the bronze table is still valid */
SELECT supplier_name, supplier_type, supplier_country, supplier_city FROM bronze_supplier.supplier_schema.bronze_supplier_basic_csv


## Use a newly created silver table and populate it from rows from bronze ## 

In [1]:
%sql
select supplier_name  from bronze_supplier.supplier_schema.bronze_supplier_basic_csv;
delete from silver_supplier.supplier_schema.silver_supplier_basic_cont_csv ;

INSERT INTO silver_supplier.supplier_schema.silver_supplier_basic_cont_csv  (supplier_name_silver, supplier_type_silver, supplier_country_silver, supplier_city_silver)
SELECT supplier_name, supplier_type, supplier_country, supplier_city FROM bronze_supplier.supplier_schema.bronze_supplier_basic_csv




OK

In [1]:
spark.sql("select * from silver_supplier.supplier_schema.silver_supplier_basic_cont_csv  ").show(50, False)

+-----------------------------------+--------------------+-----------------------+--------------------+---------+
|supplier_name_silver               |supplier_type_silver|supplier_country_silver|supplier_city_silver|continent|
+-----------------------------------+--------------------+-----------------------+--------------------+---------+
|24/7 Community Hospital Group      |NULL                |United States          |Glendale            |NULL     |
|A1 Juice Supply                    |NULL                |United States          |Aurora              |NULL     |
|ABC Bank                           |NULL                |United States          |Redwood City        |NULL     |
|ABC Consulting                     |NULL                |Australia              |Sydney              |NULL     |
|Abbott                             |NULL                |United States          |Lake Forest         |NULL     |
|Adams Leasing                      |NULL                |United States          |Greenw